# Random Annotation Inspection

Inspect randomly selected video–annotation pairs and compare visible behavior with the annotated action intervals. This notebook reads the dataset without modifying videos, annotations, or class definitions.

**Workflow:** run all cells once, then rerun the final code cell to draw a new random sample. Each result contains a full video player, an annotation table, and sampled frames labeled with the actions active at their timestamps.

Use the same Python environment as the classification demo, such as **Python (AnimalAct)**. No model or pretrained weights are required. Automatic checks assess file consistency and annotation structure; visual inspection is required to judge whether a behavior label is correct.

## 1. Configuration and Dataset Loading

`N_SAMPLES` controls how many videos are inspected per draw. Leave `VIDEO_ID = None` for random sampling, or enter a video ID to inspect a specific pair. `FRAMES_PER_VIDEO` controls the frame grid. Set `SHOW_VIDEO = False` to omit embedded video players and reduce notebook size.

Sampling uses system randomness without a fixed seed. Videos are unique within a draw, but can appear again in a later draw. The manifest links each selected video to its annotation file.
Use `SPLIT` to inspect one fixed partition. When `VIDEO_ID` is set, it must belong to that partition.


In [ ]:
import html
import json
import math
import random
import textwrap
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Video, display

N_SAMPLES = 3
FRAMES_PER_VIDEO = 6
VIDEO_ID = None
SHOW_VIDEO = True
SPLIT = None  # None, "train", "valid", or "test"

cv2.setNumThreads(1)
DATASET_DIR = next(
    (path.resolve() for path in [Path(".."), Path("."), Path("new_data/AnimalAct")]
     if (path / "manifest.json").is_file()
     and (path / "annotations").is_dir()
     and (path / "videos").is_dir()),
    None,
)
if DATASET_DIR is None:
    raise FileNotFoundError("Open this notebook from the dataset demo folder.")
manifest = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
assert SPLIT in {None, "train", "valid", "test"}
samples = [s for s in manifest["samples"] if SPLIT is None or s["split"] == SPLIT]
vocabulary = json.loads((DATASET_DIR / "action_classes.json").read_text(encoding="utf-8"))
known_actions = {item["name"] for item in vocabulary["classes"]}
assert samples and len({item["video_id"] for item in samples}) == len(samples)
print(f"Available video–annotation pairs: {len(samples):,}")
print(f"Action vocabulary: {len(known_actions)} classes")

## 2. Annotation Validation

For each sampled pair, check file availability and identifiers, video metadata, action names, and interval boundaries. Frame boundaries may equal the video frame count at the end of a clip. Differences of up to one frame between timestamps and frame boundaries are tolerated for rounding.

Overlapping actions are valid and are displayed together. An interval with invalid or missing times is reported in the table but cannot contribute a frame label. An empty annotation produces an inspection warning. Passing these checks does not establish that the visible behavior has been labeled correctly.

In [ ]:
def display_table(headers, rows):
    """Render escaped values so annotation text is displayed as text."""
    head = "".join(f"<th>{html.escape(str(value))}</th>" for value in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{head}</tr></thead>"
                 f"<tbody>{body}</tbody></table></div>"))


def finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def inspect_pair(sample):
    """Return the video path, interval rows, issues, FPS, and frame count."""
    video_path = DATASET_DIR / sample["video_path"]
    annotation_path = DATASET_DIR / sample["annotation_path"]
    issues, rows = [], []
    for path in [video_path, annotation_path]:
        if path.stem != sample["video_id"]:
            issues.append(f"ERROR: File stem does not match video_id: {path.name}")
        if not path.is_file():
            issues.append(f"ERROR: Missing file: {path.name}")
    if not video_path.is_file() or not annotation_path.is_file():
        return video_path, rows, issues, 0, 0
    capture = cv2.VideoCapture(str(video_path))
    try:
        fps = capture.get(cv2.CAP_PROP_FPS)
        raw_frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        frame_count = int(raw_frame_count) if math.isfinite(raw_frame_count) else 0
        metadata_valid = capture.isOpened() and math.isfinite(fps) and fps > 0 and frame_count > 0
    finally:
        capture.release()
    if not metadata_valid:
        issues.append("ERROR: Video could not be opened or has invalid FPS/frame count.")
        return video_path, rows, issues, 0, 0
    duration = frame_count / fps
    try:
        annotation = json.loads(annotation_path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        issues.append(f"ERROR: Cannot read annotation JSON: {error}")
        return video_path, rows, issues, fps, frame_count
    if not isinstance(annotation, dict):
        issues.append("ERROR: Annotation JSON must contain an object.")
        return video_path, rows, issues, fps, frame_count
    if annotation.get("video_id") != sample["video_id"] or annotation.get("video") != video_path.name:
        issues.append("ERROR: Annotation video identifiers do not match the selected video.")
    for label, value in [("Manifest FPS", sample.get("fps")), ("Annotation FPS", annotation.get("fps"))]:
        if not finite_number(value) or not math.isclose(value, fps, rel_tol=0.001, abs_tol=0.01):
            issues.append(f"WARNING: {label} differs from decoded video FPS ({fps:.4f}).")
    if sample.get("nb_frames") != frame_count:
        issues.append("WARNING: Manifest frame count differs from video metadata.")
    if not finite_number(sample.get("duration")) or abs(sample["duration"] - duration) > 1 / fps:
        issues.append("WARNING: Manifest duration differs from video metadata by more than one frame.")
    animals = annotation.get("annotations")
    if not isinstance(animals, list):
        issues.append("ERROR: 'annotations' must be a list.")
        return video_path, rows, issues, fps, frame_count
    for animal_index, animal in enumerate(animals):
        if not isinstance(animal, dict) or not isinstance(animal.get("actions"), list):
            issues.append(f"ERROR: Invalid animal entry {animal_index}.")
            continue
        if not isinstance(animal.get("animal"), str) or not animal["animal"].strip():
            issues.append(f"ERROR: Missing animal name in entry {animal_index}.")
        for action_index, action in enumerate(animal["actions"]):
            location = f"Animal {animal_index}, action {action_index}"
            if not isinstance(action, dict):
                issues.append(f"ERROR: {location} is not an object.")
                continue
            start, end, label = action.get("start"), action.get("end"), action.get("action")
            valid_times = finite_number(start) and finite_number(end) and 0 <= start < end
            rows.append(dict(animal=animal.get("animal", "Unknown animal"), action=label,
                             start=start, end=end, start_frame=action.get("start_frame"),
                             end_frame=action.get("end_frame"), valid_times=valid_times))
            if not isinstance(label, str) or label not in known_actions:
                issues.append(f"ERROR: {location} has an unknown action name: {label!r}.")
            if not valid_times:
                issues.append(f"ERROR: {location} needs finite times with 0 <= start < end.")
                continue
            if start >= duration or end > duration + 1 / fps + 1e-6:
                issues.append(f"ERROR: {location} is outside the video duration ({duration:.3f} s).")
            elif end > duration + 1e-6:
                issues.append(f"WARNING: {location} ends slightly beyond the video (within one frame).")
            for key, seconds in [("start_frame", start), ("end_frame", end)]:
                frame = action.get(key)
                if type(frame) is not int or not 0 <= frame <= frame_count:
                    issues.append(f"ERROR: {location} has an invalid {key}.")
                elif abs(frame - seconds * fps) > 1 + 1e-6:
                    issues.append(f"WARNING: {location}: {key} differs from its timestamp by more than one frame.")
            if type(action.get("start_frame")) is int and type(action.get("end_frame")) is int:
                if not 0 <= action["start_frame"] < action["end_frame"] <= frame_count:
                    issues.append(f"ERROR: {location} has reversed, empty, or out-of-range frame boundaries.")
    if not rows:
        issues.append("WARNING: No action intervals are available for inspection.")
    if sample.get("action_count") != len(rows):
        issues.append("WARNING: Manifest action_count differs from the annotation records.")
    return video_path, rows, issues, fps, frame_count

## 3. Video and Frame Inspection

The video player shows the complete clip. Use the interval table to compare its playback time with the annotations. Browser codec support is required for playback; decoded frame previews are also provided.

The frame grid combines evenly spaced frames with randomly chosen interval midpoints. Each title gives the frame index, timestamp, and **all animal–action labels active at that time**, using `start <= time < end`. “No active annotation” means the timestamp is not covered by an interval; it is not a background-action label. Labels apply to the video timeline and are not spatial bounding boxes.

In [ ]:
def show_frames(video_path, rows, fps, frame_count, count, rng):
    count = min(count, frame_count)
    uniform_count = max(1, count // 2)
    indices = set(np.linspace(0, frame_count - 1, uniform_count, dtype=int).tolist())
    candidates = [row for row in rows if row["valid_times"]
                  and row["start"] < frame_count / fps]
    rng.shuffle(candidates)
    for row in candidates:
        midpoint = (row["start"] + min(row["end"], frame_count / fps)) / 2
        indices.add(min(frame_count - 1, int(midpoint * fps)))
        if len(indices) >= count:
            break
    while len(indices) < count:
        indices.add(rng.randrange(frame_count))
    indices = sorted(indices)[:count]
    columns = min(3, len(indices))
    fig, axes = plt.subplots(math.ceil(len(indices) / columns), columns,
                             figsize=(16, 4.8 * math.ceil(len(indices) / columns)),
                             squeeze=False, layout="constrained")
    capture = cv2.VideoCapture(str(video_path))
    decode_issues = []
    try:
        for ax in axes.flat:
            ax.axis("off")
        for ax, frame_index in zip(axes.flat, indices):
            capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = capture.read()
            if not ok:
                ax.set_title(f"Frame {frame_index}: decode failed")
                decode_issues.append(f"ERROR: Could not decode frame {frame_index}.")
                continue
            decoded_index = int(round(capture.get(cv2.CAP_PROP_POS_FRAMES))) - 1
            if decoded_index != frame_index:
                decode_issues.append(f"WARNING: Requested frame {frame_index}, decoded {decoded_index}.")
            timestamp = decoded_index / fps
            active = [f"{row['animal']} — {row['action']}" for row in rows
                      if row["valid_times"] and row["start"] <= timestamp < row["end"]]
            label = "\n".join(dict.fromkeys(active)) if active else "No active annotation"
            title = f"Frame {decoded_index} | {timestamp:.3f} s\n" + "\n".join(
                textwrap.fill(line, 48) for line in label.splitlines()
            )
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(title, fontsize=10, loc="left")
        plt.show()
    finally:
        capture.release()
        plt.close(fig)
    return decode_issues


def inspect_random_sample():
    """Draw new pairs each time this function is called."""
    if type(N_SAMPLES) is not int or N_SAMPLES < 1:
        raise ValueError("N_SAMPLES must be a positive integer.")
    if type(FRAMES_PER_VIDEO) is not int or FRAMES_PER_VIDEO < 1:
        raise ValueError("FRAMES_PER_VIDEO must be a positive integer.")
    rng = random.SystemRandom()
    if VIDEO_ID is None:
        selected = rng.sample(samples, min(N_SAMPLES, len(samples)))
    else:
        selected = [sample for sample in samples if sample["video_id"] == VIDEO_ID]
        if not selected:
            raise ValueError(f"Unknown VIDEO_ID: {VIDEO_ID}")
    summary = []
    for sample in selected:
        video_id = sample["video_id"]
        display(HTML(f"<h3>Video {html.escape(video_id)} — {html.escape(sample['split'])}</h3>"))
        print(f"Video: {sample['video_path']}\nAnnotation: {sample['annotation_path']}")
        video_path, rows, issues, fps, frame_count = inspect_pair(sample)
        if fps > 0:
            print(f"Duration: {frame_count / fps:.3f} s | FPS: {fps:.3f} | Frames: {frame_count}")
        display_table(["Animal", "Action", "Start (s)", "End (s)", "Start frame", "End frame"],
                      [[row[key] for key in ["animal", "action", "start", "end", "start_frame", "end_frame"]]
                       for row in rows])
        if fps > 0:
            if SHOW_VIDEO:
                display(Video(filename=str(video_path), embed=True, width=800,
                              html_attributes="controls preload=metadata"))
            issues.extend(show_frames(video_path, rows, fps, frame_count, FRAMES_PER_VIDEO, rng))
        errors = sum(issue.startswith("ERROR:") for issue in issues)
        warnings = sum(issue.startswith("WARNING:") for issue in issues)
        if issues:
            display_table(["Automatic check findings"], [[issue] for issue in issues])
        else:
            print("Automatic checks passed for this pair. Review the visible behavior manually.")
        summary.append([video_id, len(rows), errors, warnings])
    display(HTML("<h3>Inspection summary</h3>"))
    display_table(["Video ID", "Action intervals", "Errors", "Warnings"], summary)
    return [sample["video_id"] for sample in selected]

## 4. Random Inspection

Rerun the cell below to draw another sample. To change settings, edit and rerun the configuration cell first.

During visual review, check whether the action name matches the behavior, the start and end times cover the behavior, and simultaneous actions are represented correctly. A gap between intervals can be valid. Record a questionable video ID and its timestamp for follow-up; this notebook does not edit annotations.

In [ ]:
selected_video_ids = inspect_random_sample()
print("Selected video IDs:", ", ".join(selected_video_ids))